In [19]:
import os
import pandas as pd

uxm_results_path = "/data/dmytro/cfSortData/UXM_results"
import numpy as np

In [2]:
metadata = pd.read_csv(
    "/data/dmytro/cfSortData/GSE233417_samples/GSE233417_samples.csv"
)

In [8]:
from methyldl.deconvolution.linear_calibrator import LinearCalibrator

calibrator = LinearCalibrator()
calibrator.load_calibration_parameters(
    "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Tutorials/uxm_results_pseudobulk/uxm_linear_calibrator.npz"
)

In [3]:
cfsort_tissue_to_celltypes = {
    "adipose tissue": ["Adipocytes"],
    "vagina": ["Epid-Kerat", "Smooth-Musc"],
    "skin": ["Epid-Kerat", "Dermal-Fibro"],
    "skin exposed": ["Epid-Kerat", "Dermal-Fibro"],
    "small intestine": ["Small-Int-Ep"],
    "esophagus": ["Epid-Kerat"],
    "esophagus gast junc": ["Gastric-Ep", "Epid-Kerat"],
    "esophagus muscularis": ["Smooth-Musc"],
    "esophagus musc": ["Smooth-Musc"],
    "kidney": ["Kidney-Ep"],
    "salivary gland": ["Head-Neck-Ep"],
    "prostate": ["Prostate-Ep"],
    "breast": ["Breast-Luminal-Ep", "Breast-Basal-Ep"],
    "nerve": ["Neuron", "Oligodend"],
    "pituitary": ["Neuron"],
    "pancreas": [
        "Pancreas-Acinar",
        "Pancreas-Beta",
        "Pancreas-Alpha",
        "Pancreas-Delta",
        "Pancreas-Duct",
    ],
    "testis": ["Epid-Kerat"],  # Sertoli / seminiferous epithelium → closest proxy
    "muscle": ["Skeletal-Musc"],
    "adrenal gland": ["Endothel"],  # no perfect match; endothelial + stromal dominant
    "blood vessel": ["Endothel", "Smooth-Musc"],
    "blood vessel coronary": ["Endothel", "Smooth-Musc"],
    "heart": ["Heart-Cardio", "Heart-Fibro"],
    "heart atrial": ["Heart-Cardio"],
    "spleen": ["Blood-B", "Blood-T", "Blood-Mono+Macro"],
    "ovary": ["Ovary-Ep"],
    "bladder": ["Bladder-Ep"],
    "cervix uteri": ["Epid-Kerat"],
    "cervix uteri endocervix": ["Head-Neck-Ep"],  # columnar glandular epithelium
    "uterus": ["Smooth-Musc"],  # myometrium dominates by mass
    "fallopian tube": ["Fallopian-Ep"],
    "thyroid": ["Thyroid-Ep"],
    "colon": ["Colon-Ep", "Colon-Fibro"],
    "liver": ["Liver-Hep"],
    "stom": ["Gastric-Ep"],  # stomach
    "lung": ["Lung-Ep-Alveo", "Lung-Ep-Bron"],
    "WBC": ["Blood-T", "Blood-B", "Blood-Mono+Macro", "Blood-NK", "Blood-Granul"],
}

In [23]:
results_top25 = []
results_top250 = []
for file in os.listdir(uxm_results_path):
    sub = pd.read_csv(os.path.join(uxm_results_path, file))
    if "Atlas.U250" in file:
        sub.columns = ["CellType", "UXM_U250"]
    else:
        sub.columns = ["CellType", "UXM_U25"]
        sub["UXM_U25_LCal"] = calibrator.predict(
            np.reshape(sub[[sub.columns[1]]].to_numpy(), (1, 39))
        )[0][0]
    sub["file"] = file.split("_")[0]
    sub["Biosample organism"] = "Homo sapiens"
    sub["Biosample type"] = "tissue"
    sub["Biosample term id"] = "not mapped"
    sub_meta = metadata[metadata["sample_geo_accession"] == file.split("_")[0]]
    if len(sub_meta) != 1:
        raise ValueError(f"Corrupted metadata for {file}")
    tissue = sub_meta["tissue"].to_list()[0]
    sub["Biosample term name"] = sub_meta["tissue"].to_list()[0]
    sub["AuditGood"] = True
    sub["CellTypeProxy"] = [cfsort_tissue_to_celltypes[tissue]] * 39
    sub["TagFiltered"] = True
    if "Atlas.U250" in file:
        res_column = "uxm_U25"
        results_top250.append(sub)
    else:
        res_column = "uxm_U25"
        results_top25.append(sub)

In [24]:
results_top25_pd = pd.concat(results_top25, axis=0).reset_index()
results_top250_pd = pd.concat(results_top250, axis=0).reset_index()

In [25]:
results_uxm_all = pd.merge(
    results_top25_pd,
    results_top250_pd[["CellType", "file", "UXM_U250"]],
    on=["CellType", "file"],
)

In [26]:
results_uxm_all.drop("index", axis=1, inplace=True)

In [28]:
results_uxm_all.to_csv("cfSort_rrbs_uxm_results.csv", index=False)